<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms import Resize, CenterCrop, ToTensor, Normalize, Compose
from PIL import Image
from collections import deque

# 1. Clone the repository if it doesn't exist
REPO_DIR = "imagenet-sample-images"
REPO_URL = "https://github.com/EliSchwartz/imagenet-sample-images.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# =====================================================================
# 2. Define the Compression Interface
# =====================================================================
class CompressionInterface(nn.Module):
    """
    Standardized interface for compression methods.
    Inherits from nn.Module to support future trainable PyTorch coders.
    """
    def __init__(self):
        super().__init__()

    def compress(self, x: torch.Tensor, level: int) -> torch.Tensor:
        raise NotImplementedError

    def decompress(self, x: torch.Tensor, level: int, target_shape: tuple) -> torch.Tensor:
        raise NotImplementedError

    def train_coder(self, dataloader):
        """Placeholder for future training logic."""
        pass

# =====================================================================
# 3. Implement a Basic Non-Trained Compression Method
# =====================================================================
class SpatialDownsamplingCoder(CompressionInterface):
    """
    A basic non-trained coder that downsamples feature maps spatially
    to reduce transmission size, and upsamples them back at the receiver.
    """
    def compress(self, x: torch.Tensor, level: int) -> torch.Tensor:
        if level == 1:
            return x
        # Scale factor decreases exponentially with compression level
        scale_factor = 1.0 / (2 ** (level - 1))
        return F.interpolate(x, scale_factor=scale_factor, mode='bilinear', align_corners=False)

    def decompress(self, x: torch.Tensor, level: int, target_shape: tuple) -> torch.Tensor:
        if level == 1:
            return x
        # Upsample back to the exact target shape
        return F.interpolate(x, size=(target_shape[2], target_shape[3]), mode='bilinear', align_corners=False)

# =====================================================================
# 4. Define the Edge (Communication Channel) Module
# =====================================================================
class CommunicationEdge(nn.Module):
    def __init__(self, source_id: str, target_id: str, bandwidth_bps: float = 10e6):
        super().__init__()
        self.source_id = source_id
        self.target_id = target_id
        self.bandwidth_bps = bandwidth_bps  # Time-constant bandwidth in bits per second

        # Compression configuration
        self.coder = SpatialDownsamplingCoder()
        self.compression_level = 1

        # Transmission time tracker (in seconds)
        self.accumulated_tx_time = 0.0

    def reset_tracker(self):
        self.accumulated_tx_time = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Save original shape for decompression
        original_shape = x.shape

        # 1. Compress
        compressed_x = self.coder.compress(x, self.compression_level)

        # 2. Calculate Transmission Time
        # Assuming 32-bit (4 bytes) float elements
        total_bits = compressed_x.numel() * 32
        tx_time = total_bits / self.bandwidth_bps
        self.accumulated_tx_time += tx_time

        # 3. Decompress
        decompressed_x = self.coder.decompress(compressed_x, self.compression_level, original_shape)
        return decompressed_x

# =====================================================================
# 5. Define the Device Node Module
# =====================================================================
class DeviceNode(nn.Module):
    def __init__(self, node_id: str, weight: float):
        super().__init__()
        self.node_id = node_id
        self.weight = weight
        self.layers = nn.Sequential()
        self.edges = nn.ModuleDict()  # Outgoing edges
        self.predecessors = []        # List of predecessor node IDs
        self.downsample_factor = 1    # Estimated during compilation

    def add_successor(self, successor_node: 'DeviceNode', bandwidth_bps: float):
        edge_id = f"{self.node_id}->{successor_node.node_id}"
        self.edges[edge_id] = CommunicationEdge(self.node_id, successor_node.node_id, bandwidth_bps)
        successor_node.predecessors.append(self.node_id)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

# =====================================================================
# 6. Define the General Distributed Graph Coordinator
# =====================================================================
class DistributedGraph(nn.Module):
    def __init__(self, bandwidth_bps: float = 10e6):
        super().__init__()
        self.nodes = nn.ModuleDict()
        self.topological_levels = []
        self.bandwidth_bps = bandwidth_bps

    def add_node(self, node_id: str, weight: float):
        self.nodes[node_id] = DeviceNode(node_id, weight)

    def add_edge(self, source_id: str, target_id: str):
        self.nodes[source_id].add_successor(self.nodes[target_id], self.bandwidth_bps)

    def set_compression_level(self, level: int):
        """Sets the compression level across all communication edges."""
        for node in self.nodes.values():
            for edge in node.edges.values():
                edge.compression_level = level

    def get_cumulative_tx_time(self) -> float:
        """Sums the transmission times across all edges in the graph."""
        total_time = 0.0
        for node in self.nodes.values():
            for edge in node.edges.values():
                total_time += edge.accumulated_tx_time
        return total_time

    def reset_tx_trackers(self):
        for node in self.nodes.values():
            for edge in node.edges.values():
                edge.reset_tracker()

    def _get_in_channels(self, module: nn.Module) -> int:
        for sub_module in module.modules():
            if isinstance(sub_module, nn.Conv2d):
                return sub_module.in_channels
        return 3

    def compile_and_allocate(self, model: nn.Module):
        in_degree = {node_id: len(node.predecessors) for node_id, node in self.nodes.items()}
        queue = deque([node_id for node_id, deg in in_degree.items() if deg == 0])

        levels = []
        while queue:
            level_nodes = list(queue)
            levels.append(level_nodes)
            queue.clear()
            for node_id in level_nodes:
                for edge in self.nodes[node_id].edges.values():
                    target_id = edge.target_id
                    in_degree[target_id] -= 1
                    if in_degree[target_id] == 0:
                        queue.append(target_id)

        if sum(len(lvl) for lvl in levels) != len(self.nodes):
            raise ValueError("Graph contains cycles.")

        self.topological_levels = levels
        D = len(levels)

        features_layers = list(model.features.children())
        num_layers = len(features_layers)

        block_size = num_layers // D
        for d, level_nodes in enumerate(levels):
            start_idx = d * block_size
            end_idx = (d + 1) * block_size if d < D - 1 else num_layers
            layer_block = nn.Sequential(*features_layers[start_idx:end_idx])

            for node_id in level_nodes:
                node = self.nodes[node_id]
                if d == D - 1:
                    node.layers = nn.Sequential(
                        layer_block,
                        model.avgpool,
                        nn.Flatten(1),
                        model.classifier
                    )
                else:
                    node.layers = layer_block

        with torch.no_grad():
            for node in self.nodes.values():
                in_channels = self._get_in_channels(node.layers)
                dummy_in = torch.randn(1, in_channels, 224, 224)
                try:
                    test_layers = node.layers[0] if isinstance(node.layers[0], nn.Sequential) else node.layers
                    dummy_out = test_layers(dummy_in)
                    node.downsample_factor = dummy_in.shape[2] // dummy_out.shape[2]
                except Exception:
                    node.downsample_factor = 1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        node_outputs = {}

        for d, level_nodes in enumerate(self.topological_levels):
            for node_id in level_nodes:
                node = self.nodes[node_id]

                if d == 0:
                    node_input = x
                else:
                    incoming_tensors = []
                    for pred_id in node.predecessors:
                        edge_id = f"{pred_id}->{node_id}"
                        transmitted_tensor = self.nodes[pred_id].edges[edge_id](node_outputs[pred_id])
                        incoming_tensors.append(transmitted_tensor)

                    if len(incoming_tensors) == 1:
                        node_input = incoming_tensors[0]
                    else:
                        scale = node.downsample_factor
                        pad_size = self.nodes[node.predecessors[0]].downsample_factor
                        crop_rows = pad_size // scale

                        h_A = incoming_tensors[0].shape[2] - crop_rows
                        clean_A = incoming_tensors[0][:, :, 0:h_A, :]
                        clean_B = incoming_tensors[1][:, :, crop_rows:, :]
                        node_input = torch.cat([clean_A, clean_B], dim=2)

                node_outputs[node_id] = node(node_input)

                if len(node.edges) > 1:
                    successors = [self.nodes[edge.target_id] for edge in node.edges.values()]
                    total_weight = sum(succ.weight for succ in successors)

                    B, C, H, W = node_outputs[node_id].shape
                    h_A = int(round((successors[0].weight / total_weight) * H))

                    pad_size = successors[0].downsample_factor
                    slice_A = node_outputs[node_id][:, :, 0 : h_A + pad_size, :]
                    slice_B = node_outputs[node_id][:, :, h_A - pad_size : H, :]

                    node_outputs[node_id] = slice_A
                    node_outputs[f"{node_id}_split2"] = slice_B

        terminal_node_id = self.topological_levels[-1][0]
        return node_outputs[terminal_node_id]

# =====================================================================
# 7. Self-Contained ImageNet Loader
# =====================================================================
def load_imagenet_samples(k=20):
    all_files = [f for f in os.listdir(REPO_DIR) if f.endswith(".JPEG")]
    unique_wnids = sorted(list(set(f.split("_")[0] for f in all_files)))
    wnid_to_idx = {wnid: idx for idx, wnid in enumerate(unique_wnids)}

    selected_files = random.sample(all_files, min(k, len(all_files)))

    transform = Compose([
        Resize(224),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    images, targets, names = [], [], []
    for filename in selected_files:
        parts = filename.replace(".JPEG", "").split("_")
        wnid = parts[0]
        class_name = " ".join(parts[1:])
        class_idx = wnid_to_idx[wnid]
        img_path = os.path.join(REPO_DIR, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(transform(img))
            targets.append(class_idx)
            names.append(class_name)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return torch.stack(images), torch.tensor(targets), names

# =====================================================================
# 8. Main Evaluation Pipeline
# =====================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation on: {device}\n")

    # Load Pretrained VGG16
    print("Loading pretrained VGG16 model...")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device).eval()

    # Define DAG Topology with a constant bandwidth of 10 Mbps (10e6 bps)
    bandwidth_bps = 10e6
    dist_graph = DistributedGraph(bandwidth_bps=bandwidth_bps)
    dist_graph.add_node('P', weight=1.0)
    dist_graph.add_node('A', weight=0.6)
    dist_graph.add_node('B', weight=0.4)
    dist_graph.add_node('S', weight=1.0)

    dist_graph.add_edge('P', 'A')
    dist_graph.add_edge('P', 'B')
    dist_graph.add_edge('A', 'S')
    dist_graph.add_edge('B', 'S')

    # Compile and allocate layers automatically
    dist_graph.compile_and_allocate(vgg16)
    dist_graph.to(device)

    # Load k random ImageNet samples
    k_samples = 50
    print(f"Loading {k_samples} random ImageNet samples from local repository...")
    images, targets, names = load_imagenet_samples(k=k_samples)
    images, targets = images.to(device), targets.to(device)

    # --- EVALUATION 1: Baseline (Unsplit) ---
    print("\nRunning Baseline (Unsplit) inference...")
    with torch.no_grad():
        out_base = vgg16(images)

    correct_base = sum(out_base[i].argmax().item() == targets[i].item() for i in range(k_samples))
    acc_base = (correct_base / k_samples) * 100

    # --- EVALUATION 2: Distributed Model with No Compression (Level 1) ---
    print("Running Distributed Graph (No Compression) inference...")
    dist_graph.set_compression_level(1)
    dist_graph.reset_tx_trackers()

    with torch.no_grad():
        out_dag_no_comp = dist_graph(images)

    tx_time_no_comp = dist_graph.get_cumulative_tx_time()
    correct_dag_no_comp = sum(out_dag_no_comp[i].argmax().item() == targets[i].item() for i in range(k_samples))
    acc_no_comp = (correct_dag_no_comp / k_samples) * 100

    # --- EVALUATION 3: Distributed Model with Increasing Compression Levels ---
    compression_results = []
    for level in [2, 3, 4]:
        print(f"Running Distributed Graph (Compression Level {level}) inference...")
        dist_graph.set_compression_level(level)
        dist_graph.reset_tx_trackers()

        with torch.no_grad():
            out_dag_comp = dist_graph(images)

        tx_time_comp = dist_graph.get_cumulative_tx_time()
        correct_dag_comp = sum(out_dag_comp[i].argmax().item() == targets[i].item() for i in range(k_samples))
        acc_comp = (correct_dag_comp / k_samples) * 100

        compression_results.append({
            'level': level,
            'accuracy': acc_comp,
            'tx_time': tx_time_comp
        })

    # --- PRINT FINAL COMPARISON TABLE ---
    print("\n" + "="*90)
    print(f"EVALUATION RESULTS ON {k_samples} IMAGENET SAMPLES (Bandwidth: {bandwidth_bps/1e6:.1f} Mbps)")
    print("="*90)
    print(f"{'Execution Mode':<35} | {'Accuracy (%)':<15} | {'Cumulative Tx Time (s)':<25}")
    print("-"*90)
    print(f"{'Baseline (Unsplit Model)':<35} | {acc_base:<15.2f} | {'N/A (Local)':<25}")
    print(f"{'Distributed (No Compression - Lvl 1)':<35} | {acc_no_comp:<15.2f} | {tx_time_no_comp:<25.4f}")

    for res in compression_results:
        mode_str = f"Distributed (Compression Level {res['level']})"
        print(f"{mode_str:<35} | {res['accuracy']:<15.2f} | {res['tx_time']:<25.4f}")
    print("="*90)

if __name__ == "__main__":
    main()

Running evaluation on: cpu

Loading pretrained VGG16 model...
Loading 50 random ImageNet samples from local repository...

Running Baseline (Unsplit) inference...
Running Distributed Graph (No Compression) inference...
Running Distributed Graph (Compression Level 2) inference...
Running Distributed Graph (Compression Level 3) inference...
Running Distributed Graph (Compression Level 4) inference...

EVALUATION RESULTS ON 50 IMAGENET SAMPLES (Bandwidth: 10.0 Mbps)
Execution Mode                      | Accuracy (%)    | Cumulative Tx Time (s)   
------------------------------------------------------------------------------------------
Baseline (Unsplit Model)            | 88.00           | N/A (Local)              
Distributed (No Compression - Lvl 1) | 74.00           | 165.1507                 
Distributed (Compression Level 2)   | 34.00           | 41.2877                  
Distributed (Compression Level 3)   | 4.00            | 9.7485                   
Distributed (Compression Level